In [1]:
import pandas as pd
import joblib
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from utils import fit_position_pipeline, select_features_by_correlation

In [2]:
# features = joblib.load('../predictors/hist/feats_mid')
features = [
    "was_home",
    "difficulty",
    "opponent_difficulty",
    "win_prob",
    "elo_diff",
    "yellow_cards_rolling_5",
    "starts_rolling_3",
    "points_3_ewm",
    "xP_3_ewm",
    "minutes_rolling_3",
    "goals_scored_3_ewm",
    "assists_3_ewm",
    "clean_sheets_3_ewm",
    "goals_conceded_3_ewm",
    "ict_index_3_ewm",
    "expected_goals_3_ewm",
    "expected_assists_3_ewm",
    "expected_goal_involvements_3_ewm",
    "expected_goals_conceded_3_ewm",
    "interceptions_3_ewm",
    "blocks_3_ewm",
    "clearances_3_ewm",
    "tackles_3_ewm",
    "shots_faced_3_ewm",
    "goals_prevented_3_ewm",
    "selected_by_percent",
    "transfers_in",
]

player_data = pd.read_csv("../../inputs/rolled_data_24-25_xp_fixed.csv")
player_data = player_data[player_data["position"] == "Midfielder"]

league_models = {}

for gw in range(6, 39):
    print(f"\n{'=' * 60}\n Model for round {gw} \n{'=' * 60}")
    data = player_data[(player_data["round"] > 3) & (player_data["round"] < gw)].fillna(
        0
    )
    feats = features  # features[gw]
    # feats.remove('xP')
    selected_feats = select_features_by_correlation(
        data,
        feats,
        target_col="xP",
        redundancy_threshold=0.45,
        min_corr=0.15,
        verbose=True,
    )

    league_model = fit_position_pipeline(
        data, selected_feats, gw, position_name="MID", random_seed=42
    )

    print(f"Features for MID kept in gw {gw} -> {league_model['feats']}")

    league_models[gw] = league_model



 Model for round 19 
  dropping 'xP_3_ewm' (|corr| with xP=0.855) -- |corr|=0.877 with already-selected 'points_3_ewm'
  dropping 'ict_index_3_ewm' (|corr| with xP=0.816) -- |corr|=0.909 with already-selected 'points_3_ewm'
  dropping 'expected_goal_involvements_3_ewm' (|corr| with xP=0.760) -- |corr|=0.831 with already-selected 'points_3_ewm'
  dropping 'minutes_rolling_3' (|corr| with xP=0.691) -- |corr|=0.771 with already-selected 'points_3_ewm'
  dropping 'expected_goals_3_ewm' (|corr| with xP=0.677) -- |corr|=0.753 with already-selected 'points_3_ewm'
  dropping 'goals_scored_3_ewm' (|corr| with xP=0.662) -- |corr|=0.784 with already-selected 'points_3_ewm'
  dropping 'starts_rolling_3' (|corr| with xP=0.651) -- |corr|=0.736 with already-selected 'points_3_ewm'
  dropping 'expected_assists_3_ewm' (|corr| with xP=0.622) -- |corr|=0.663 with already-selected 'points_3_ewm'
  dropping 'selected_by_percent' (|corr| with xP=0.572) -- |corr|=0.540 with already-selected 'points_3_ewm'
 

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

... (192 outputs left out)

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  divergences: 0
Features for MID kept in gw 38 -> ['elo_diff', 'yellow_cards_rolling_5', 'points_3_ewm', 'clearances_3_ewm']


In [3]:
joblib.dump(league_models, "./league_models_mid")

['./league_models_mid']